# 比賽影片自動標註工具（在 Colab 上跑）

在 Google 的 GPU 上啟動隊友的標註介面（含我們加的可信度功能），然後給你一個公開網址，用 Mac 的瀏覽器打開就能用。

**每次使用：**
1. 上方選單「執行階段 → 變更執行階段類型」→ 確認是 **T4 GPU**
2. 「執行階段 → 全部執行」
3. 等 3～5 分鐘，最後一格會出現 `https://xxxx.gradio.live` 的網址，點開就是介面

**注意：** 網址只在這個 Colab 分頁開著的時候有效。關掉分頁、或閒置太久被 Colab 斷線，就要重新執行一次，網址也會換一個新的。

## 1. 確認有 GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "沒有 GPU：請到「執行階段 → 變更執行階段類型」選 T4 GPU，然後重新執行"
print("GPU OK:", torch.cuda.get_device_name(0))

## 2. 安裝工具和模型（約 2～3 分鐘）

In [ ]:
%cd /content
!rm -rf TEED-match-tagger
!git clone -q https://github.com/jmoussa99/TEED-match-tagger.git
!git clone -q https://github.com/arturxe2/T-DEED TEED-match-tagger/private/T-DEED
!git -C TEED-match-tagger/private/T-DEED checkout -q d953b4de8a2ad941c565e1683c5b07255cbe644e
!git -C TEED-match-tagger/private/T-DEED apply /content/TEED-match-tagger/app/patches/tdeed_mps.patch
!pip -q install gradio==6.31.0 timm gdown
CKPT_DIR = "/content/TEED-match-tagger/private/T-DEED/checkpoints/SoccerNetBall/SoccerNetBall_challenge2"
!mkdir -p {CKPT_DIR} /content/TEED-match-tagger/reports/gar_eval_v1
!gdown -q 1uCWam8EhAXt3z6QQAfhRfzHITUcml438 -O {CKPT_DIR}/checkpoint_best.pt
!ls -lh {CKPT_DIR}

## 3. 加上我們的改動（可信度功能和 Colab 啟動腳本；還沒推上 GitHub，所以直接寫進來）

In [ ]:
%%writefile /content/TEED-match-tagger/app/ui.py
"""Gradio app: tag ball actions in a full match with T-DEED.

Run from the repository root:
    .venv/bin/python app/ui.py        # then open http://127.0.0.1:7860
"""

from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import gradio as gr
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
import reliability as rel  # noqa: E402
import tdeed_runner as tr  # noqa: E402

PREVIEW_DIR = tr.CACHE_DIR / "previews"
SERIES_COLOR = "#2a78d6"  # single series; class identity is carried by the axis, not color
PREVIEW_HALF_WINDOW = 4.0  # seconds before/after the event

TABLE_COLUMNS = ["time", "label", "confidence", "reliability", "seconds"]


def _resolve_video(upload: str | None, local_path: str) -> Path:
    if local_path and local_path.strip():
        p = Path(local_path.strip().strip("'\"")).expanduser()
        if not p.is_file():
            raise gr.Error(f"File not found: {p}")
        return p
    if upload:
        return Path(upload)
    raise gr.Error("Upload a video or paste a local file path.")


def _views(scores, video: str, threshold: float, classes: list[str]):
    """Build table, charts, summary and downloads from cached scores (no model call)."""
    if scores is None:
        empty = pd.DataFrame(columns=TABLE_COLUMNS)
        return empty, None, None, "No match tagged yet.", None, None
    events = tr.extract_events(scores, threshold, classes)
    df = pd.DataFrame(events, columns=["time", "seconds", "position_ms", "label", "confidence"])
    df["reliability"] = df["label"].map(rel.level)
    json_path, csv_path = tr.save_events(events, video, threshold)

    counts = pd.DataFrame({
        "label": classes,
        "events": [int((df["label"] == c).sum()) for c in classes],
        "series": "Events",
    }).sort_values("events", ascending=False)
    bar = gr.BarPlot(
        counts, x="label", y="events", color="series", color_map={"Events": SERIES_COLOR},
        colors_in_legend=[], sort="-y", x_title="", y_title="Events", x_label_angle=-35,
        tooltip=["label", "events"], height=320, label="Events per class",
    )

    timeline_df = df.assign(minute=df["seconds"] / 60.0, series="Events")
    timeline = gr.ScatterPlot(
        timeline_df, x="minute", y="label", color="series", color_map={"Events": SERIES_COLOR},
        colors_in_legend=[], x_title="Match time (minutes)", y_title="",
        tooltip=["time", "label", "confidence"], height=360, label="Timeline",
    )

    minutes = len(scores) / tr.SAMPLED_FPS / 60
    summary = (f"**{len(df)} events** in {minutes:.1f} min of video "
               f"(confidence ≥ {threshold:.2f}, {len(classes)} classes shown).")
    shaky = df[df["reliability"].isin([rel.LOW, rel.UNTESTED])]
    if len(shaky):
        names = ", ".join(sorted(shaky["label"].unique()))
        summary += (f"\n\n⚠️ {len(shaky)} of these events are from classes with **low or untested reliability** "
                    f"({names}). Check them in the preview before relying on them.")
    return df[TABLE_COLUMNS], bar, timeline, summary, str(json_path), str(csv_path)


def tag_match(upload, local_path, threshold, classes, progress=gr.Progress()):
    video = _resolve_video(upload, local_path)

    def report(frac: float, msg: str) -> None:
        progress(frac, desc=msg)

    try:
        scores = tr.compute_scores(video, progress=report)
    except FileNotFoundError as e:
        raise gr.Error(str(e)) from e
    except RuntimeError as e:
        raise gr.Error(f"Could not read the video: {e}") from e
    return (scores, str(video), *_views(scores, str(video), threshold, classes))


def refilter(scores, video, threshold, classes):
    return _views(scores, video, threshold, classes)


def preview(video: str | None, evt: gr.SelectData):
    if not video or evt.row_value is None:
        return None
    seconds = float(evt.row_value[TABLE_COLUMNS.index("seconds")])
    start = max(seconds - PREVIEW_HALF_WINDOW, 0.0)
    PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
    out = PREVIEW_DIR / f"{Path(video).stem}_{int(seconds * 1000)}.mp4"
    if not out.is_file():
        subprocess.run(
            ["ffmpeg", "-v", "error", "-y", "-ss", f"{start:.2f}", "-i", video,
             "-t", f"{2 * PREVIEW_HALF_WINDOW:.2f}", "-vf", "scale=-2:480",
             "-c:v", "libx264", "-preset", "veryfast", "-crf", "26", "-pix_fmt", "yuv420p",
             "-c:a", "aac", "-movflags", "+faststart", str(out)],
            check=True,
        )
    return str(out)


def build() -> gr.Blocks:
    with gr.Blocks(title="Match Event Tagger") as demo:
        gr.Markdown(
            "# Match event tagger\n"
            "Automatic ball-action tagging with T-DEED (SoccerNet Ball Action Spotting). "
            "The first run on a match takes a while; afterwards changing the threshold or classes is instant."
        )
        scores_state = gr.State(None)
        video_state = gr.State(None)

        with gr.Row():
            with gr.Column(scale=2):
                upload = gr.File(label="Upload match video", file_types=["video"], type="filepath")
                local_path = gr.Textbox(
                    label="…or local file path (faster for large files, no upload copy)",
                    placeholder="/Users/you/Movies/match.mp4",
                )
                run_btn = gr.Button("Tag match", variant="primary")
            with gr.Column(scale=1):
                threshold = gr.Slider(0.05, 0.95, value=tr.DEFAULT_THRESHOLD, step=0.05,
                                      label="Minimum confidence")
                classes = gr.CheckboxGroup(tr.CLASSES, value=tr.CLASSES, label="Classes")
                with gr.Row():
                    all_btn = gr.Button("All 12 classes", size="sm")
                    gar_btn = gr.Button("GAR 10 classes", size="sm")
                    reliable_btn = gr.Button("Reliable classes only", size="sm")
                with gr.Accordion("How reliable is each class?", open=False):
                    gr.Markdown(rel.legend_markdown(tr.CLASSES))

        summary = gr.Markdown("No match tagged yet.")
        with gr.Row():
            with gr.Column(scale=3):
                table = gr.Dataframe(headers=TABLE_COLUMNS, interactive=False, max_height=520,
                                     label="Events (click a row to preview)", show_search="filter")
            with gr.Column(scale=2):
                clip = gr.Video(label="Event preview", interactive=False, autoplay=True)
                with gr.Row():
                    json_file = gr.File(label="Events JSON", interactive=False)
                    csv_file = gr.File(label="Events CSV", interactive=False)
        with gr.Row():
            bar = gr.BarPlot(label="Events per class")
            timeline = gr.ScatterPlot(label="Timeline")

        view_outputs = [table, bar, timeline, summary, json_file, csv_file]
        run_btn.click(tag_match, [upload, local_path, threshold, classes],
                      [scores_state, video_state, *view_outputs])
        for control in (threshold, classes):
            control.change(refilter, [scores_state, video_state, threshold, classes], view_outputs)
        all_btn.click(lambda: tr.CLASSES, None, classes)
        gar_btn.click(lambda: tr.GAR_CLASSES, None, classes)
        reliable_btn.click(lambda: rel.classes_at_least(tr.CLASSES, rel.MEDIUM), None, classes)
        table.select(preview, [video_state], clip)
    return demo


if __name__ == "__main__":
    build().queue(default_concurrency_limit=1).launch(
        server_name="127.0.0.1", allowed_paths=[str(tr.REPO_ROOT / "outputs")],
    )

In [ ]:
%%writefile /content/TEED-match-tagger/app/reliability.py
"""How much to trust each T-DEED class on footage it was not trained on.

T-DEED was trained on SoccerNet broadcasts (English league). We measured it on 436 labeled
clips of a different broadcast source (SoccerNet-GAR validation, World Cup 2022 footage from
PFF): see reports/gar_eval_v1/README.md. The recall per class there is the share of real
events of that class that the tagger labels correctly.

Levels (recall): high >= 0.70, medium >= 0.40, low < 0.40. DRIVE and BALL PLAYER BLOCK do not
exist in GAR, so they are untested.

The numbers are read from reports/gar_eval_v1/per_class.csv when it exists (so a newer
evaluation updates the app automatically); otherwise the values below are used.
"""
from __future__ import annotations

import csv
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parents[1]
RESULTS_CSV = REPO_ROOT / "reports" / "gar_eval_v1" / "per_class.csv"

# Recall on GAR validation clips (reports/gar_eval_v1, 50 clips per class; 23 goals, 13 high passes).
_FALLBACK_RECALL = {
    "PASS": 0.86, "OUT": 0.76, "SHOT": 0.74, "THROW IN": 0.60, "HEADER": 0.50, "CROSS": 0.30,
    "PLAYER SUCCESSFUL TACKLE": 0.0, "FREE KICK": 0.0, "GOAL": 0.0, "HIGH PASS": 0.0,
}

HIGH, MEDIUM, LOW, UNTESTED = "high", "medium", "low", "untested"
_ORDER = {HIGH: 0, MEDIUM: 1, LOW: 2, UNTESTED: 3}


def _load_recall() -> dict[str, float]:
    if RESULTS_CSV.is_file():
        with RESULTS_CSV.open() as f:
            rows = {r["class"]: float(r["recall"]) for r in csv.DictReader(f) if r.get("recall")}
        if rows:
            return rows
    return dict(_FALLBACK_RECALL)


RECALL = _load_recall()


def level(label: str) -> str:
    r = RECALL.get(label)
    if r is None:
        return UNTESTED
    return HIGH if r >= 0.70 else MEDIUM if r >= 0.40 else LOW


def classes_at_least(classes: list[str], min_level: str = MEDIUM) -> list[str]:
    """Classes (kept in the given order) whose reliability is min_level or better."""
    return [c for c in classes if _ORDER[level(c)] <= _ORDER[min_level]]


def legend_markdown(classes: list[str]) -> str:
    rows = sorted(classes, key=lambda c: (_ORDER[level(c)], -(RECALL.get(c) or 0)))
    lines = [
        "Measured on labeled clips from a **different broadcast source** than T-DEED was trained on "
        "(World Cup 2022 footage, SoccerNet-GAR). *Recall* = share of real events of that class the tagger "
        "labels correctly. Treat **low** classes as unreliable: most real tackles, free kicks, goals and "
        "high passes are missed or labeled as something else (goals usually appear as SHOT).",
        "",
        "| Class | Reliability | Recall |",
        "|---|---|---|",
    ]
    for c in rows:
        r = RECALL.get(c)
        lines.append(f"| {c} | {level(c)} | {'—' if r is None else f'{r:.0%}'} |")
    return "\n".join(lines)

In [ ]:
%%writefile /content/TEED-match-tagger/reports/gar_eval_v1/per_class.csv
class,n_clips,recall,precision
PASS,50,0.86,0.232
HEADER,50,0.5,0.581
HIGH PASS,13,0.0,0.0
OUT,50,0.76,0.463
CROSS,50,0.3,0.714
THROW IN,50,0.6,0.968
SHOT,50,0.74,0.561
PLAYER SUCCESSFUL TACKLE,50,0.0,
FREE KICK,50,0.0,
GOAL,23,0.0,

In [ ]:
%%writefile /content/TEED-match-tagger/scripts/evaluate_gar.py
"""Evaluate the T-DEED tagger on labeled SoccerNet-GAR video clips.

Every GAR clip is 4.8 s long and its labeled action happens at ~2.43 s (the clip center).
For each clip we run the same scoring pipeline as the app (app/tdeed_runner.compute_scores),
take each class's highest score within +/- WINDOW_S seconds of the action time, and predict
the class with the highest score. Predictions are restricted to the 10 GAR classes by default
(T-DEED's extra classes DRIVE and BALL PLAYER BLOCK do not exist in GAR).

Usage (from this repo's root, after scripts/setup_tdeed.sh):
  .venv/bin/python scripts/evaluate_gar.py                       # every clip found in --videos
  .venv/bin/python scripts/evaluate_gar.py --per-class 20        # at most 20 clips per class
  .venv/bin/python scripts/evaluate_gar.py --limit 10            # quick smoke test

Outputs (outputs/gar_eval/): predictions.csv, per_class.csv, confusion.csv, metrics.json
"""
from __future__ import annotations

import argparse
import json
import random
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT / "app"))
import tdeed_runner as tr  # noqa: E402

if sys.platform != "darwin":
    # The app decodes with macOS VideoToolbox ("-hwaccel videotoolbox"), which does not exist on
    # Linux (e.g. Google Colab). Re-use the app's decoder with that flag removed.
    import subprocess as _sp
    _orig_popen = _sp.Popen

    def _popen_without_videotoolbox(cmd, *args, **kwargs):
        if isinstance(cmd, list) and "videotoolbox" in cmd:
            i = cmd.index("videotoolbox")
            cmd = cmd[: i - 1] + cmd[i + 1:]
        return _orig_popen(cmd, *args, **kwargs)

    tr.subprocess.Popen = _popen_without_videotoolbox

GAR_PROJECT = REPO_ROOT.parent / "sn-gar-2027"
ACTION_TIME_S = 2.43   # alignment.action_offset_seconds in the GAR video metadata
WINDOW_S = 1.0         # +/- 1 s, the same tolerance as the SoccerNet mAP@1 metric
OUT_DIR = REPO_ROOT / "outputs" / "gar_eval"


def load_labels(annotations: Path) -> dict[str, str]:
    data = json.load(open(annotations))["data"]
    return {s["id"]: s["labels"]["action"]["label"] for s in data}


def clip_id(path: Path, split: str) -> str:
    # clip_000123.mp4 -> valid_000123
    return f"{split}_{path.stem.split('_')[-1]}"


def predict_from_scores(scores: np.ndarray, classes: list[str]) -> tuple[str, float, dict[str, float]]:
    """scores: (N, 1 + 12) per-frame probabilities at tr.SAMPLED_FPS (column 0 = background)."""
    center = int(round(ACTION_TIME_S * tr.SAMPLED_FPS))
    half = int(round(WINDOW_S * tr.SAMPLED_FPS))
    lo, hi = max(0, center - half), min(len(scores), center + half + 1)
    window = scores[lo:hi, 1:]                      # drop background
    peak = window.max(axis=0) if len(window) else np.zeros(len(tr.CLASSES))
    per_class = {c: float(peak[i]) for i, c in enumerate(tr.CLASSES)}
    best = max(classes, key=lambda c: per_class[c])
    return best, per_class[best], per_class


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--split", default="valid", choices=["train", "valid", "test"])
    ap.add_argument("--annotations", type=Path, default=None,
                    help="GAR annotations JSON (default: ../sn-gar-2027/data/annotations_<split>.json)")
    ap.add_argument("--videos", type=Path, default=None,
                    help="folder of GAR clips (default: ../sn-gar-2027/data/videos/<split>)")
    ap.add_argument("--per-class", type=int, default=None, help="max clips per class")
    ap.add_argument("--limit", type=int, default=None, help="max clips in total")
    ap.add_argument("--all-12", action="store_true", help="allow DRIVE / BALL PLAYER BLOCK predictions")
    ap.add_argument("--device", default="auto", choices=["auto", "cuda", "mps", "cpu"],
                    help="auto = cuda if available, else mps, else cpu")
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()

    annotations = a.annotations or GAR_PROJECT / "data" / f"annotations_{a.split}.json"
    videos_dir = a.videos or GAR_PROJECT / "data" / "videos" / a.split
    labels = load_labels(annotations)

    clips = sorted(videos_dir.glob("clip_*.mp4"))
    clips = [p for p in clips if clip_id(p, a.split) in labels]
    if not clips:
        sys.exit(f"No labeled clips found in {videos_dir}")
    if a.per_class:
        rng = random.Random(a.seed)
        by_class: dict[str, list[Path]] = {}
        for p in clips:
            by_class.setdefault(labels[clip_id(p, a.split)], []).append(p)
        clips = sorted(p for ps in by_class.values() for p in rng.sample(ps, min(a.per_class, len(ps))))
    if a.limit:
        clips = clips[: a.limit]
    print(f"{len(clips)} clips:", dict(Counter(labels[clip_id(p, a.split)] for p in clips).most_common()))

    classes = tr.CLASSES if a.all_12 else tr.GAR_CLASSES
    device = a.device
    if device == "auto" and tr.torch.cuda.is_available():
        device = "cuda"
    model = tr.get_model(device)
    print(f"Model on {tr.pick_device(device)}")
    rows, t0 = [], time.perf_counter()
    for i, p in enumerate(clips, 1):
        scores = tr.compute_scores(p, model=model)
        pred, conf, per_class = predict_from_scores(scores, classes)
        cid = clip_id(p, a.split)
        rows.append({"id": cid, "label": labels[cid], "pred": pred, "confidence": round(conf, 4),
                     **{f"score_{c}": round(v, 4) for c, v in per_class.items()}})
        if i % 10 == 0 or i == len(clips):
            el = time.perf_counter() - t0
            print(f"  {i}/{len(clips)} clips · {el / i:.1f} s per clip · ETA {el / i * (len(clips) - i) / 60:.1f} min",
                  flush=True)

    df = pd.DataFrame(rows)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT_DIR / "predictions.csv", index=False)

    present = [c for c in tr.GAR_CLASSES if (df.label == c).any()]
    recall = {c: float((df[df.label == c].pred == c).mean()) for c in present}
    precision = {c: float((df[df.pred == c].label == c).mean()) if (df.pred == c).any() else float("nan")
                 for c in present}
    per_class = pd.DataFrame({"class": present,
                              "n_clips": [int((df.label == c).sum()) for c in present],
                              "recall": [round(recall[c], 3) for c in present],
                              "precision": [round(precision[c], 3) for c in present]})
    per_class.to_csv(OUT_DIR / "per_class.csv", index=False)
    pd.crosstab(df.label, df.pred).to_csv(OUT_DIR / "confusion.csv")
    metrics = {"split": a.split, "n_clips": len(df), "classes_present": len(present),
               "prediction_classes": "all 12" if a.all_12 else "GAR 10",
               "window_seconds": WINDOW_S, "action_time_seconds": ACTION_TIME_S,
               "balanced_accuracy": 100 * float(np.mean(list(recall.values()))),
               "accuracy": 100 * float((df.label == df.pred).mean()),
               "seconds_per_clip": round((time.perf_counter() - t0) / len(df), 2)}
    json.dump(metrics, open(OUT_DIR / "metrics.json", "w"), indent=2)

    print(json.dumps(metrics, indent=2))
    print(per_class.to_string(index=False))
    print(f"\nSaved to {OUT_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/TEED-match-tagger/scripts/launch_colab.py
"""Run the match-tagger web UI on Google Colab (NVIDIA GPU) with a public share link.

The app is written for macOS: it decodes video with VideoToolbox and picks MPS or CPU.
This launcher adapts both at runtime without editing the app:
  - ffmpeg's "-hwaccel videotoolbox" is dropped on Linux (via scripts/evaluate_gar.py's patch)
  - "auto" device selection prefers CUDA when available
then starts the same Gradio UI with share=True, which prints a https://xxxx.gradio.live link.

Usage (from the repo root on Colab):  python scripts/launch_colab.py [--extra-path /content/drive]
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import evaluate_gar as eg  # noqa: E402  (imports app/tdeed_runner and applies the Linux ffmpeg fix)

tr = eg.tr
_orig_pick_device = tr.pick_device


def _pick_device(name: str = "auto"):
    if name == "auto" and tr.torch.cuda.is_available():
        return tr.torch.device("cuda")
    return _orig_pick_device(name)


tr.pick_device = _pick_device

import ui  # noqa: E402  (app/ui.py, already on sys.path via evaluate_gar)


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--extra-path", action="append", default=[],
                    help="folder the UI may read videos from via the local-path box (e.g. /content/drive)")
    a = ap.parse_args()
    print(f"Model will run on: {tr.pick_device('auto')}")
    ui.build().queue(default_concurrency_limit=1).launch(
        share=True,
        allowed_paths=[str(tr.REPO_ROOT / "outputs"), *a.extra_path],
    )


if __name__ == "__main__":
    main()

## 4.（選用）連接 Google Drive

比賽影片通常很大，從瀏覽器上傳會很慢。可以先把影片放進你的 Google Drive，執行這格後，
在介面的「local file path」欄位貼上路徑，例如 `/content/drive/MyDrive/match.mp4`。
用不到的話可以跳過這格。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 5. 啟動介面

等到下面出現 **`Running on public URL: https://xxxx.gradio.live`**，點開那個網址。
這格會一直顯示執行中，這是正常的；要關掉介面就按這格左邊的停止鍵。

In [ ]:
%cd /content/TEED-match-tagger
!python scripts/launch_colab.py --extra-path /content/drive